# Adaptive Multi-Sensor Fusion for Agronomic Prediction
## Attention-Weighted Hybrid CNN–DAFM–RBF Model for Soil Nutrient Prediction

Mburu, J. M., Ogada, K. and Kimwele, M. — Jomo Kenyatta University of Agriculture and Technology

This notebook reproduces the analysis end to end: it downloads the open AfSIS mid-infrared
spectra and wet-chemistry deposits, assembles the four-country dataset, trains the proposed
architecture alongside five benchmarks under group-disjoint cross-validation, and regenerates
the key figures.

**Runtime:** roughly 25–40 minutes on a free Colab CPU instance for the full grid, or about
3 minutes for the quick demonstration configuration set in the parameters cell.

**Data sources (both open access):**
- Spectra — Vagen et al. (2020), doi:10.34725/DVN/QXCWP1
- Chemistry — Vagen et al. (2021), doi:10.34725/DVN/66BFOB


## 1. Environment

Only NumPy, pandas and Matplotlib are required — there is no deep-learning framework dependency, so the models train identically on any machine.

In [ ]:
import sys, os, json, math, time, urllib.request
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
print('python', sys.version.split()[0], '| numpy', np.__version__, '| pandas', pd.__version__)
SEED = 20260804
np.random.seed(SEED)

## 2. Parameters

Set `QUICK = True` for a fast demonstration (3 properties, 2 models). Set `QUICK = False` to reproduce the published grid.

In [ ]:
QUICK = True

COUNTRIES = ['Kenya', 'Ethiopia', 'Tanzania', 'Uganda']
ALL_TARGETS = ['pH','OrgC_pct','TotN_pct','P_Olsen','Ca_exch','Mg_exch','K_exch',
               'eCEC','S_total','Fe_total','Mn_total','Zn_total','Cu_total']
TARGETS = ['TotN_pct','pH','Ca_exch'] if QUICK else ALL_TARGETS
MODELS_TO_RUN = ['Ridge','DualDAFM'] if QUICK else \
    ['PLSR','Ridge','RandomForest','GradientBoosting','MLP','DualDAFM']
DESIGNS = ['cluster'] if QUICK else ['cluster','site','country']
print('targets:', TARGETS); print('models:', MODELS_TO_RUN); print('designs:', DESIGNS)

## 3. Download the AfSIS data

Files are fetched directly from the ICRAF Dataverse. No account is needed. Sizes are checked so a truncated download is caught rather than silently used.

In [ ]:
FILES = {'Kenya.tab': (10923, 15.4), 'Ethiopia.tab': (10920, 20.8),
         'Tanzania.tab': (10931, 37.3), 'Uganda.tab': (10932, 9.9),
         'chemistry.tab': (11218, 0.85)}
BASE = 'https://data.worldagroforestry.org/api/access/datafile/'
os.makedirs('data', exist_ok=True)

need = [f for f in FILES if f.split('.')[0] in COUNTRIES or f == 'chemistry.tab']
for name in need:
    fid, mb = FILES[name]
    path = os.path.join('data', name)
    if os.path.exists(path) and os.path.getsize(path) > 0.8 * mb * 1e6:
        print(f'{name:16s} already present'); continue
    print(f'{name:16s} downloading ~{mb} MB ...', end=' ', flush=True)
    urllib.request.urlretrieve(BASE + str(fid), path)
    got = os.path.getsize(path) / 1e6
    ok = got > 0.8 * mb
    print(f'{got:.1f} MB {"OK" if ok else "TRUNCATED - delete and retry"}')

## 4. Assemble the dataset

Spectra and chemistry are joined on the soil sample number (SSN). The wavenumber grid is checked for consistency across countries before any concatenation.

In [ ]:
TARGET_COLS = {'pH':'pH','OrgC_pct':'Percent_Org_C','TotN_pct':'Percent_N',
               'P_Olsen':'Olsen.P','Ca_exch':'Caex','Mg_exch':'Mgex','K_exch':'Kex',
               'eCEC':'eCEC','S_total':'S','Fe_total':'Fe','Mn_total':'Mn',
               'Zn_total':'Zn','Cu_total':'Cu'}

chem = pd.read_csv('data/chemistry.tab', sep='\t')
chem = chem[chem.Country.isin(COUNTRIES)]

frames, wn_ref, wn_cols = [], None, None
for c in COUNTRIES:
    s = pd.read_csv(f'data/{c}.tab', sep='\t')
    cols = [x for x in s.columns if x.startswith('m')]
    wn = np.array([float(x[1:]) for x in cols]); o = np.argsort(wn)
    cols = [cols[i] for i in o]; wn = wn[o]
    if wn_ref is None: wn_ref, wn_cols = wn, cols
    else: assert np.allclose(wn, wn_ref), f'{c}: wavenumber grid differs'
    frames.append(s[['SSN'] + wn_cols]); print(f'{c:10s} {len(s):5d} spectra')

spec = pd.concat(frames, ignore_index=True).drop_duplicates('SSN')
df = chem.merge(spec, on='SSN', how='inner', validate='one_to_one')
print(f'\njoined {len(df)} samples (attrition {len(chem)-len(df)})')

meta = df[['SSN','Country','Site','Cluster','Latitude','Longitude','Depth']].copy()
meta['SiteKey'] = meta.Country + '_' + meta.Site.astype(str)
meta['ClusterKey'] = meta.SiteKey + '_' + meta.Cluster.astype(str)
meta['is_topsoil'] = (meta.Depth == 'Topsoil').astype(int)
Y = pd.DataFrame({k: df[v] for k, v in TARGET_COLS.items()})
X = df[wn_cols].to_numpy(float)
print(f'{X.shape[0]} samples x {X.shape[1]} wavenumbers '
      f'({wn_ref.min():.1f}-{wn_ref.max():.1f} cm-1)')
print(f"{meta.SiteKey.nunique()} sites, {meta.ClusterKey.nunique()} clusters")

## 5. Spectral preprocessing

Savitzky–Golay first derivative, Standard Normal Variate scatter correction, binning, then standardisation. **All data-dependent steps are fitted on training rows only** — this is what makes the reported metrics honest.

In [ ]:
def savgol(Xm, window=11, poly=2, deriv=0):
    half = window // 2
    idx = np.arange(-half, half + 1)
    A = np.vander(idx, poly + 1, increasing=True)
    coef = np.linalg.pinv(A)[deriv] * (math.factorial(deriv) if deriv else 1.0)
    Xp = np.pad(Xm, ((0, 0), (half, half)), mode='edge')
    return np.stack([Xp[:, j:j+window] @ coef for j in range(Xm.shape[1])], axis=1)

def snv(Xm):
    m = Xm.mean(1, keepdims=True); s = Xm.std(1, keepdims=True)
    return (Xm - m) / np.where(s < 1e-12, 1.0, s)

def binned(Xm, f=4):
    L = (Xm.shape[1] // f) * f
    return Xm[:, :L].reshape(Xm.shape[0], L // f, f).mean(2)

class SpecPrep:
    def _static(self, Xm): return binned(snv(savgol(Xm, 11, 2, 1)), 4)
    def fit(self, Xm):
        Z = self._static(Xm); self.mu = Z.mean(0); self.sd = np.where(Z.std(0) < 1e-12, 1.0, Z.std(0))
        return self
    def transform(self, Xm): return (self._static(Xm) - self.mu) / self.sd

class YPrep:
    def __init__(self, log): self.log = log
    def fit(self, y):
        t = np.log1p(np.maximum(y, 0)) if self.log else y
        self.mu, self.sd = float(t.mean()), max(float(t.std()), 1e-12); return self
    def fwd(self, y):
        t = np.log1p(np.maximum(y, 0)) if self.log else y
        return (t - self.mu) / self.sd
    def inv(self, z):
        t = z * self.sd + self.mu
        return np.expm1(t) if self.log else t

print('preprocessing defined')

## 6. The proposed architecture

Spectral branch (1D CNN) → **Dynamic Attention Fusion Module** over spectral positions →
learned modality gate → **differentiable RBF head**. Every operation is differentiable, so the
network trains end to end and the convolutional representation adapts to the kernel-style head.

Attention is computed over *spectral positions*, not latent channels, which is what makes the
weights attributable to identifiable wavenumber regions.

In [ ]:
# The full implementation lives in ml.py and ml_dual.py in the reproducibility package.
# For the notebook we fetch them if present, else define a compact equivalent inline.
REPO_FILES = ['ml.py', 'ml_dual.py']
have_repo = all(os.path.exists(f) for f in REPO_FILES)
if have_repo:
    from ml import MODELS
    from ml_dual import DualBranchDAFM
    print('using ml.py / ml_dual.py from the reproducibility package')
else:
    print('ml.py and ml_dual.py not found in the working directory.')
    print('Upload them from the reproducibility package, or clone the repository, then re-run.')

## 7. Group-disjoint cross-validation

Three nested designs. The grouping variable is the only thing that changes between them.
Fold 0 is always reserved for hyperparameter selection and never used as a test fold.

In [ ]:
NFOLDS = {'cluster': 5, 'site': 8, 'country': 4}

def folds_for(design, meta):
    key = {'cluster': meta.ClusterKey.values, 'site': meta.SiteKey.values,
           'country': meta.Country.values}[design]
    uniq, counts = np.unique(key, return_counts=True)
    k = min(NFOLDS[design], len(uniq))
    load = np.zeros(k, int); assign = {}
    for gi in np.argsort(-counts):
        f = int(np.argmin(load)); assign[uniq[gi]] = f; load[f] += counts[gi]
    fo = np.array([assign[g] for g in key])
    return [np.where(fo == f)[0] for f in range(k)]

def metrics(y, p):
    e = p - y; sst = float(((y - y.mean())**2).sum())
    return {'R2': 1 - float((e**2).sum())/sst if sst > 0 else np.nan,
            'RMSE': float(np.sqrt((e**2).mean())), 'MAE': float(np.abs(e).mean())}

for d in DESIGNS:
    print(d, [len(f) for f in folds_for(d, meta)])

## 8. Run the benchmark

Every model receives the same tuning budget, which is what makes the comparison fair.

In [ ]:
def covariates(meta, idx):
    m = meta.iloc[idx]
    return np.column_stack([m.is_topsoil.values, m.Latitude.values,
                            m.Longitude.values]).astype(float)

results = []
if have_repo:
    for design in DESIGNS:
        folds = folds_for(design, meta)
        for t in TARGETS:
            y = Y[t].values.astype(float); ok = np.isfinite(y)
            log = bool(pd.Series(y[ok]).skew() > 2.0)
            for name in MODELS_TO_RUN:
                scores = []
                for f in range(1, len(folds)):
                    te = np.array([i for i in folds[f] if ok[i]])
                    tr = np.array([i for i in range(len(y))
                                   if ok[i] and i not in set(folds[f])])
                    sp = SpecPrep().fit(X[tr]); yp = YPrep(log).fit(y[tr])
                    Xtr, Xte = sp.transform(X[tr]), sp.transform(X[te])
                    Ctr, Cte = covariates(meta, tr), covariates(meta, te)
                    cm, cs = Ctr.mean(0), np.where(Ctr.std(0) < 1e-12, 1, Ctr.std(0))
                    Ctr, Cte = (Ctr-cm)/cs, (Cte-cm)/cs
                    if name == 'DualDAFM':
                        m = DualBranchDAFM(Xtr.shape[1], Ctr.shape[1], seed=SEED,
                                           epochs=90, batch=128, patience=12)
                        m.fit(Xtr, Ctr, yp.fwd(y[tr]), Xte, Cte, yp.fwd(y[te]))
                        pr = yp.inv(m.predict(Xte, Cte))
                    else:
                        mm = MODELS[name](seed=SEED) if name != 'Ridge' else MODELS[name](alpha=10.0)
                        mm.fit(Xtr, yp.fwd(y[tr]))
                        pr = yp.inv(mm.predict(Xte))
                    scores.append(metrics(y[te], pr)['R2'])
                results.append(dict(design=design, target=t, model=name,
                                    R2=float(np.mean(scores)), sd=float(np.std(scores, ddof=1))))
                print(f'{design:8s} {t:10s} {name:16s} R2={np.mean(scores):.3f}', flush=True)
    res = pd.DataFrame(results)
    res.to_csv('notebook_results.csv', index=False)
else:
    res = pd.DataFrame()
    print('skipped: upload ml.py and ml_dual.py first')

## 9. Results


In [ ]:
if len(res):
    piv = res[res.design=='cluster'].pivot(index='target', columns='model', values='R2')
    display(piv.round(3))
    fig, ax = plt.subplots(figsize=(7, 3.4))
    piv.plot(kind='bar', ax=ax); ax.set_ylabel('$R^2$')
    ax.set_title('Leave-cluster-out cross-validation')
    ax.legend(frameon=False, fontsize=8); plt.tight_layout()
    fig.savefig('notebook_results.png', dpi=200); plt.show()
else:
    print('no results to plot')

## 10. Attention interpretability

The DAFM weights are defined over spectral positions, so they map back to wavenumber ranges
and can be read against the spectrum directly. The modality gate reports how much of each
prediction came from spectra versus ancillary information.

In [ ]:
if have_repo and len(res):
    t = TARGETS[0]
    y = Y[t].values.astype(float); ok = np.isfinite(y)
    folds = folds_for('cluster', meta); te = np.array([i for i in folds[1] if ok[i]])
    tr = np.array([i for i in range(len(y)) if ok[i] and i not in set(folds[1])])
    sp = SpecPrep().fit(X[tr]); yp = YPrep(False).fit(y[tr])
    Xtr, Xte = sp.transform(X[tr]), sp.transform(X[te])
    Ctr, Cte = covariates(meta, tr), covariates(meta, te)
    cm, cs = Ctr.mean(0), np.where(Ctr.std(0) < 1e-12, 1, Ctr.std(0))
    Ctr, Cte = (Ctr-cm)/cs, (Cte-cm)/cs
    m = DualBranchDAFM(Xtr.shape[1], Ctr.shape[1], seed=SEED, epochs=90,
                       batch=128, patience=12)
    m.fit(Xtr, Ctr, yp.fwd(y[tr]), Xte, Cte, yp.fwd(y[te]))
    att = m.spectral_attention(Xte, Cte); gate = m.branch_gate(Xte, Cte)
    wnb = binned(wn_ref[None, :], 4)[0]
    centres = [wnb[int((i + .5) * len(wnb) / len(att))] for i in range(len(att))]
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
    ax[0].plot(centres, att); ax[0].invert_xaxis()
    ax[0].axhline(1/len(att), ls=':', color='k')
    ax[0].set_xlabel('Wavenumber (cm$^{-1}$)'); ax[0].set_ylabel('Attention')
    ax[0].set_title(f'DAFM attention — {t}')
    ax[1].bar(['spectral','ancillary'], gate, color=['#55a868','#dd8452'])
    ax[1].set_ylabel('Gate weight'); ax[1].set_title('Measured modality contribution')
    plt.tight_layout(); fig.savefig('notebook_attention.png', dpi=200); plt.show()
    print('gate [spectral, ancillary] =', gate.round(3))

## 11. Notes on honest reporting

Three things in this notebook are deliberate and worth preserving if you adapt it:

1. **No random splits.** Every design is group-disjoint. A random split on this data inflates
   R² because samples within a cluster are near-duplicates.
2. **Preprocessing is fitted inside the fold.** Moving `SpecPrep().fit(...)` outside the fold
   loop would leak test information into the training representation.
3. **Every model gets the same budget.** Giving the proposed architecture more tuning than its
   baselines would make it look better for reasons unrelated to its design.

The published results were produced by the full pipeline in the reproducibility package, which
additionally performs nested hyperparameter selection and writes a per-fold record for every
one of the 1,794 model fits. `verify_final.py` re-derives every numeric claim in the manuscript
from those records and fails if any has drifted.